In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
file_path = "../data/raw/dynamic_supply_chain_logistics_dataset_with_country.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (113097, 18)


In [3]:
df_clean = df.copy()

print("Working copy created.")

Working copy created.


In [4]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df_clean.columns.tolist())

['warehouse_inventory_level', 'handling_equipment_availability', 'order_fulfillment_status', 'weather_condition_severity', 'shipping_costs', 'supplier_reliability_score', 'lead_time_days', 'historical_demand', 'cargo_condition_status', 'route_risk_level', 'customs_clearance_time', 'disruption_likelihood_score', 'delay_probability', 'risk_classification', 'delivery_time_deviation', 'product_id', 'supplier_id', 'supplier_country']


In [5]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113097 entries, 0 to 113096
Data columns (total 18 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   warehouse_inventory_level        113097 non-null  float64
 1   handling_equipment_availability  113097 non-null  float64
 2   order_fulfillment_status         113097 non-null  float64
 3   weather_condition_severity       113097 non-null  float64
 4   shipping_costs                   113097 non-null  float64
 5   supplier_reliability_score       113097 non-null  float64
 6   lead_time_days                   113097 non-null  float64
 7   historical_demand                113097 non-null  float64
 8   cargo_condition_status           113097 non-null  float64
 9   route_risk_level                 113097 non-null  float64
 10  customs_clearance_time           113097 non-null  float64
 11  disruption_likelihood_score      113097 non-null  float64
 12  de

In [6]:
print("Missing values:")
print(df_clean.isnull().sum())

Missing values:
warehouse_inventory_level          0
handling_equipment_availability    0
order_fulfillment_status           0
weather_condition_severity         0
shipping_costs                     0
supplier_reliability_score         0
lead_time_days                     0
historical_demand                  0
cargo_condition_status             0
route_risk_level                   0
customs_clearance_time             0
disruption_likelihood_score        0
delay_probability                  0
risk_classification                0
delivery_time_deviation            0
product_id                         0
supplier_id                        0
supplier_country                   0
dtype: int64


In [7]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 0


In [8]:
numeric_cols = df_clean.select_dtypes(include=np.number).columns

for col in numeric_cols:
    print(
        col,
        "Min:", df_clean[col].min(),
        "Max:", df_clean[col].max()
    )

warehouse_inventory_level Min: 1.32e-12 Max: 999.9992985
handling_equipment_availability Min: 4.57e-16 Max: 0.999999492
order_fulfillment_status Min: 1.32e-06 Max: 1.0
weather_condition_severity Min: 4.54e-09 Max: 0.999999997
shipping_costs Min: 100.0 Max: 999.9998534
supplier_reliability_score Min: 6.9e-10 Max: 0.999999989
lead_time_days Min: 1.0 Max: 14.99999548
historical_demand Min: 100.0029656 Max: 10000.0
cargo_condition_status Min: 7.26e-19 Max: 0.999999983
route_risk_level Min: 4.97e-05 Max: 10.0
customs_clearance_time Min: 0.500000001 Max: 4.999999928
disruption_likelihood_score Min: 4.77e-05 Max: 1.0
delay_probability Min: 3.13e-06 Max: 1.0
delivery_time_deviation Min: -1.999997989 Max: 10.0


In [9]:
outlier_summary = {}

for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df_clean[
        (df_clean[col] < lower) |
        (df_clean[col] > upper)
    ]

    outlier_summary[col] = len(outliers)

print("Outlier Counts:")
for col, count in outlier_summary.items():
    print(col, ":", count)

Outlier Counts:
warehouse_inventory_level : 0
handling_equipment_availability : 0
order_fulfillment_status : 0
weather_condition_severity : 0
shipping_costs : 0
supplier_reliability_score : 0
lead_time_days : 0
historical_demand : 0
cargo_condition_status : 0
route_risk_level : 0
customs_clearance_time : 0
disruption_likelihood_score : 8562
delay_probability : 0
delivery_time_deviation : 0


In [10]:
target_related_columns = [
    "delay_probability",
    "risk_classification",
    "delivery_time_deviation",
    "disruption_likelihood_score"
]

for col in target_related_columns:
    print("\n", col)

    if col in df_clean.columns:
        print(df_clean[col].describe(include="all"))


 delay_probability
count    113097.000000
mean          0.699617
std           0.324297
min           0.000003
25%           0.456054
50%           0.840807
75%           0.982401
max           1.000000
Name: delay_probability, dtype: float64

 risk_classification
count        113097
unique            3
top       High Risk
freq          84368
Name: risk_classification, dtype: object

 delivery_time_deviation
count    113097.000000
mean          5.170553
std           4.158751
min          -1.999998
25%           1.259607
50%           6.088516
75%           9.247708
max          10.000000
Name: delivery_time_deviation, dtype: float64

 disruption_likelihood_score
count    113097.000000
mean          0.803447
std           0.279062
min           0.000048
25%           0.691751
50%           0.958108
75%           0.998764
max           1.000000
Name: disruption_likelihood_score, dtype: float64


In [11]:
# Clean categorical text columns

categorical_cols = df_clean.select_dtypes(include="object").columns

for col in categorical_cols:
    df_clean[col] = df_clean[col].str.strip()

print("Categorical columns cleaned successfully!")

for col in categorical_cols:
    print(col, ":", df_clean[col].nunique(), "unique values")

Categorical columns cleaned successfully!
risk_classification : 3 unique values
product_id : 1000 unique values
supplier_id : 3524 unique values
supplier_country : 94 unique values


In [12]:
# Check risk classification categories

print(df_clean["risk_classification"].value_counts())
print("\nPercentages:")
print(
    df_clean["risk_classification"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

risk_classification
High Risk        84368
Moderate Risk    17783
Low Risk         10946
Name: count, dtype: int64

Percentages:
risk_classification
High Risk        74.60
Moderate Risk    15.72
Low Risk          9.68
Name: proportion, dtype: float64


In [13]:
# Validate expected numerical ranges

range_checks = {
    "handling_equipment_availability": (0, 1),
    "order_fulfillment_status": (0, 1),
    "weather_condition_severity": (0, 1),
    "supplier_reliability_score": (0, 1),
    "cargo_condition_status": (0, 1),
    "disruption_likelihood_score": (0, 1),
    "delay_probability": (0, 1),
    "route_risk_level": (0, 10),
    "customs_clearance_time": (0, 5),
    "lead_time_days": (0, 15)
}

for col, (minimum, maximum) in range_checks.items():

    invalid = df_clean[
        (df_clean[col] < minimum) |
        (df_clean[col] > maximum)
    ]

    print(f"{col}: {len(invalid)} invalid values")

handling_equipment_availability: 0 invalid values
order_fulfillment_status: 0 invalid values
weather_condition_severity: 0 invalid values
supplier_reliability_score: 0 invalid values
cargo_condition_status: 0 invalid values
disruption_likelihood_score: 0 invalid values
delay_probability: 0 invalid values
route_risk_level: 0 invalid values
customs_clearance_time: 0 invalid values
lead_time_days: 0 invalid values


In [14]:
# Check ID columns

print("Unique Products:", df_clean["product_id"].nunique())
print("Unique Suppliers:", df_clean["supplier_id"].nunique())
print("Unique Supplier Countries:", df_clean["supplier_country"].nunique())

print("\nSample Product IDs:")
print(df_clean["product_id"].head())

print("\nSample Supplier IDs:")
print(df_clean["supplier_id"].head())

Unique Products: 1000
Unique Suppliers: 3524
Unique Supplier Countries: 94

Sample Product IDs:
0    P0353
1    P0353
2    P0353
3    P0857
4    P0857
Name: product_id, dtype: object

Sample Supplier IDs:
0    P0353_S1
1    P0353_S2
2    P0353_S3
3    P0857_S1
4    P0857_S2
Name: supplier_id, dtype: object


In [15]:
# Save the final cleaned dataset

output_path = "../data/processed/supply_chain_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File location:", output_path)
print("Final dataset shape:", df_clean.shape)

Cleaned dataset saved successfully!
File location: ../data/processed/supply_chain_cleaned.csv
Final dataset shape: (113097, 18)


In [16]:
# Final preprocessing verification

print("=" * 55)
print("FINAL PREPROCESSING REPORT")
print("=" * 55)

print("Final Shape:", df_clean.shape)
print("Total Rows:", df_clean.shape[0])
print("Total Columns:", df_clean.shape[1])

print("\nTotal Missing Values:", df_clean.isnull().sum().sum())
print("Total Duplicate Rows:", df_clean.duplicated().sum())

print("\nNumerical Columns:",
      len(df_clean.select_dtypes(include=np.number).columns))

print("Categorical Columns:",
      len(df_clean.select_dtypes(include="object").columns))

print("\nRisk Classification Distribution:")
print(df_clean["risk_classification"].value_counts())

print("\nUnique Products:", df_clean["product_id"].nunique())
print("Unique Suppliers:", df_clean["supplier_id"].nunique())
print("Unique Countries:", df_clean["supplier_country"].nunique())

print("\n" + "=" * 55)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 55)

FINAL PREPROCESSING REPORT
Final Shape: (113097, 18)
Total Rows: 113097
Total Columns: 18

Total Missing Values: 0
Total Duplicate Rows: 0

Numerical Columns: 14
Categorical Columns: 4

Risk Classification Distribution:
risk_classification
High Risk        84368
Moderate Risk    17783
Low Risk         10946
Name: count, dtype: int64

Unique Products: 1000
Unique Suppliers: 3524
Unique Countries: 94

DATA PREPROCESSING COMPLETED SUCCESSFULLY
